In [2]:
import csv
import math
import regex
import os

In [7]:
log_dir_path = '/data/ros2/ros2_ws2/arm_bot/src/scripts/logs'
source_dataset_dir_path = '/data/ros2/ros2_ws2/arm_bot/src/scripts/Joint_states'
dataset_dir_path = '/data/ros2/ros2_ws2/arm_bot/src/scripts/dataset'

log_filename_pattern = regex.compile(r'trajectory_log_(\d+)_(\d{8}_\d{6})\.csv')
source_filename_pattern = regex.compile(r'path_(\d+)_joint_states.csv')
dataset_filename_prefix = 'path_'

In [ ]:
# compare log files and source dataset files to find matches
log_files = {}
for filename in os.listdir(log_dir_path):
    match = log_filename_pattern.match(filename)
    if match:
        path_id = match.group(1)
        timestamp = match.group(2)
        log_files[path_id] = (filename, timestamp)

source_files = {}
for filename in os.listdir(source_dataset_dir_path):
    match = source_filename_pattern.match(filename)
    if match:
        path_id = int(match.group(1))
        source_files[path_id] = filename

for files_id in log_files:
    log_filename, timestamp = log_files[files_id]
    print(f'Processing log file: {log_filename} with timestamp: {timestamp}')

print('\n===================================================================================\n')
for files_id in source_files:
    source_filename = source_files[files_id]
    print(f'Processing source dataset file: {source_filename} with path ID: {files_id}')

In [ ]:
# This script processes trajectory logs and corresponding joint state files to create a cropped dataset for training. It matches log files with their corresponding joint state files based on the path ID, extracts relevant data, and saves it in a new format suitable for training machine learning models.

def load_data_from_log(log_file_path):
    data = []
    with open(log_file_path, 'r') as log_file:
        csv_reader = csv.reader(log_file)
        next(csv_reader)  # Skip header
        for row in csv_reader:
            timestamp = row[0]
            joint_positions = list(map(float, row[1:4]))  # joint positions are in columns 1-3
            joint_velocities = list(map(float, row[4:7]))  # joint velocities are in columns 4-6
            joint_torques = list(map(float, row[7:10]))  # joint torques are in columns 7-9
            data.append((timestamp, joint_positions, joint_velocities, joint_torques))
    return data

In [ ]:
def load_data_from_source(source_file_path):
    """
    This function extracts timestamp, joint positions, and joint velocities from the source dataset file which containes the data in transpose format.
    the values are in the following format:
    t, xxx, xxx
    dp1, xxx, xxx
    dp2, xxx, xxx
    dp3, xxx, xxx
    dv1, xxx, xxx
    dv2, xxx, xxx
    dv3, xxx, xxx
    ......
    """
    data = []
    with open(source_file_path, 'r') as source_file:
        csv_reader = csv.reader(source_file)
        rows = list(csv_reader)
        timestamps = rows[0][1:]  # First row contains timestamps
        joint_positions = [list(map(float, row[1:])) for row in rows[1:4]]  # Rows contain joint positions
        joint_velocities = [list(map(float, row[1:])) for row in rows[4:7]]  # Rows contain joint velocities
        joint_torques = [list(map(float, row[1:])) for row in rows[10:13]]  # Rows contain joint torques
        other_data = [list(map(float, row[1:])) for row in rows[7:10] + rows[13:]]  # Rows contain other data (if needed)
        for i in range(len(timestamps)):
            data.append((timestamps[i], [joint_positions[j][i] for j in range(3)], [joint_velocities[j][i] for j in range(3)], [joint_torques[j][i] for j in range(3)], [other_data[j][i] for j in range(len(other_data))]))
    return data

In [ ]:
def find_start_index(log_data, source_data):
    """
    This function finds the start index in the log data where the three consecutive joint_torques(must) and optionally three consecutive joint_positions match the source dataset values.
    the start index is obtained by taking the first three data points from source dataset and finding the first occurrence in the log data where these points match from the start 
    """
    source_joint_positions = [source_data[i][1] for i in range(3)]  # First three data points of joint positions
    source_joint_torques = [source_data[i][3] for i in range(3)]  # First three data points of joint torques

    for i in range(len(log_data) - 2):
        log_joint_positions = [log_data[i+j][1] for j in range(3)]
        log_joint_torques = [log_data[i+j][3] for j in range(3)]

        if (log_joint_torques == source_joint_torques and log_joint_positions == source_joint_positions):
            return i  # Return the start index where the match is found

    return None  # Return None if no match is found

In [ ]:
def find_end_index(log_data, start_index, source_data):
    """
    This function finds the end index in the log data where the three consecutive joint_torques(must) and optionally three consecutive joint_positions match the source dataset values.
    the end index is obtained by taking a window of 10 data points and finding the point where the log data starts to deviate from the source dataset values after the start index.
    the window should be moved in both datasets moving one box at a time.
    for example, 
    start index is 27,
    so the window should be pointing from 0-10 in source data and from 27-37 in log data, then move the window to 1-11 in source data and 28-38 in log data.
    the mismatch should be detected by comparing the values in the window and fix the window at the first point where all the 10 data points in the window do not match between the log data and the source dataset values.
    then the end index should be the last point in the log data where the match is found before the mismatch is detected.

    the torque value threshold for mismatch is 1
    the position value threshold for mismatch is 0.01
    """
    window_size = 10
    for i in range(start_index, len(log_data) - window_size):
        log_window_joint_positions = [log_data[i+j][1] for j in range(window_size)]
        log_window_joint_torques = [log_data[i+j][3] for j in range(window_size)]

        source_offset = i - start_index
        source_window_joint_positions = [source_data[source_offset + j][1] for j in range(window_size)]
        source_window_joint_torques = [source_data[source_offset + j][3] for j in range(window_size)]

        if (log_window_joint_torques != source_window_joint_torques and log_window_joint_positions != source_window_joint_positions):
            return i - 1 # Return the last index where the match is found before the mismatch

    return len(log_data) - 1  # Return the last index if no mismatch is found

In [ ]:
def crop_log_data(log_data, start_index, end_index):
    return log_data[start_index:end_index+1]

In [ ]:
def save_cropped_dataset(cropped_data, output_file_path):
    """
    This function saves the cropped dataset.
    the cropped data should have other data from the source dataset as well.
    """
    with open(output_file_path, 'w', newline='') as output_file:
        csv_writer = csv.writer(output_file)
        # Write header
        csv_writer.writerow(['t', 'dp1', 'dp2', 'dp3', 'dv1', 'dv2', 'dv3', 'tau1', 'tau2', 'tau3', 'da1', 'da2', 'da3', 'm1', 'm2', 'm3', 'c1', 'c2', 'c3', 'g1', 'g2', 'g3'])
        # Write data rows
        for entry in cropped_data:
            timestamp = entry[0]
            joint_positions = entry[1]
            joint_velocities = entry[2]
            joint_torques = entry[3]
            other_data = entry[4]
            csv_writer.writerow([timestamp] + joint_positions + joint_velocities + joint_torques + other_data)

In [ ]:
import pandas as pd

# Read the CSV into a DataFrame
df = pd.read_csv('logs/trajectory_log_74_20260222_215656.csv', header=0)

# Rename the columns to match your desired header names
df = df.rename(columns={
    'time_elapsed': 't',
    'pos1': 'dp1',
    'pos2': 'dp2',
    'pos3': 'dp3',
    'vel1': 'dv1',
    'vel2': 'dv2',
    'vel3': 'dv3',
    'torque1': 'tau1',
    'torque2': 'tau2',
    'torque3': 'tau3'
})
df = df.T
# Write the transposed DataFrame to a new CSV file
df.to_csv('path_232_joint_states.csv', header=False, index=True) 
df.head()

,0,1,2,3,4,5,6,7,8,9,...,217,218,219,220,221,222,223,224,225,226
t,0.005000,0.016000,0.026000,0.036000,0.046000,0.056000,0.066000,0.076000,0.086000,0.096000,...,2.175000,2.185000,2.196000,2.205000,2.215000,2.225000,2.235000,2.246000,2.256000,2.265000
dp1,1.415218,1.415218,1.415218,1.415218,1.415218,1.415218,1.415218,1.415218,1.414971,1.414476,...,-0.864109,-0.862431,-0.860478,-0.858249,-0.855746,-0.852970,-0.849920,-0.846599,-0.843007,-0.839145
dp2,-2.250476,-2.250476,-2.250476,-2.250476,-2.250476,-2.250476,-2.250476,-2.250476,-2.250179,-2.249587,...,-3.216856,-3.236848,-3.256837,-3.276822,-3.296801,-3.316774,-3.336738,-3.356693,-3.376635,-3.396563
dp3,2.306014,2.306014,2.306014,2.306014,2.306013,2.306013,2.306013,2.306013,2.306116,2.306318,...,3.670599,3.662307,3.653755,3.644927,3.635803,3.626363,3.616586,3.606453,3.595939,3.585024
dv1,-0.000001,-0.000001,-0.000001,-0.000001,-0.000001,-0.000001,-0.000001,-0.000001,-0.024732,-0.049470,...,0.140004,0.167721,0.195342,0.222870,0.250308,0.277659,0.304925,0.332108,0.359212,0.386239


In [ ]:
import pandas as pd
import re
import os

# Read the CSV into a DataFrame
# log_dir_path = '/data/ros2/ros2_ws2/arm_bot/src/scripts/logs'
# dataset_dir_path = '/data/ros2/ros2_ws2/arm_bot/src/scripts/Joint_states'
# new_dataset_dir_path = '/data/ros2/ros2_ws2/arm_bot/src/scripts/new_generatedDataset'
log_dir_path = os.path.expanduser('~/Desktop/arm_bot/src/scripts/logs')
dataset_dir_path = os.path.expanduser('~/Desktop/arm_bot/src/scripts/Joint_states')
new_dataset_dir_path = os.path.expanduser('~/Desktop/arm_bot/src/scripts/new_generatedDataset')

os.makedirs(new_dataset_dir_path, exist_ok=True)

for filename in os.listdir(log_dir_path):
    match = re.search(r'path_(\d+)_log_(\d+)\.csv$', filename)
    # if match and int(match.group(1)) <= 600 and int(match.group(1)) >= 451:
    if match and int(match.group(1)) <= 601 and int(match.group(1)) >= 600:
        log_file_path = os.path.join(log_dir_path, filename)
        print(f'Processing log file: {filename}')

        df = pd.read_csv(log_file_path)

        # add acceleration fields by calculating the difference between consecutive velocity values and filling the first value with 0.
        df['da1'] = df['vel1'].diff().fillna(0).round(6)  # da1
        df['da2'] = df['vel2'].diff().fillna(0).round(6)  # da2
        df['da3'] = df['vel3'].diff().fillna(0).round(6)  # da3

        # Rename the columns to match your desired header names
        df = df.rename(columns={
            'time_elapsed': 't',
            'pos1': 'dp1',
            'pos2': 'dp2',
            'pos3': 'dp3',
            'vel1': 'dv1',
            'vel2': 'dv2',
            'vel3': 'dv3',
            'torque1': 'tau1',
            'torque2': 'tau2',
            'torque3': 'tau3',
            'da1': 'da1',
            'da2': 'da2',
            'da3': 'da3'
        })
        df = df.T
        output_file_path = os.path.join(new_dataset_dir_path, f'path_{match.group(1)}_joint_states_{match.group(2)}.csv')
        print(f'Saving processed data to: {output_file_path}')
        df.to_csv(output_file_path, header=False, index=True)
        df.head()

In [21]:
import re


log_basename = 'trajectory_path_005_log_2'

# Extract trajectory name from dataset filename (e.g., trajectory_log_path_005_2 -> path_005_2)
match = re.search(r'trajectory_(\w+)', log_basename)
if match:
    traj_name = match.group(1)
    output_file = f'src/scripts/plots/trajectory_comparison_{traj_name}_allinone.png'
else:
    output_file = f'src/scripts/plots/trajectory_comparison_allinone.png'

print(output_file)

src/scripts/plots/trajectory_comparison_path_005_log_2_allinone.png


In [23]:
import csv
import math
import os
from time import time
import pandas as pd
import os
import numpy as np

# Read the CSV into a DataFrame
dataset_file_path = "/data/ros2/ros2_ws2/arm_bot/src/scripts/Joint_states/path_678_joint_states.csv" # row wise data
log_file_path = "/data/ros2/ros2_ws2/arm_bot/src/scripts/logs/path_678_joint_states_dnn_log_2.csv"  # log file with column wise data
ctc_log_file_path = "/data/ros2/ros2_ws2/arm_bot/src/scripts/logs/path_678_joint_states_ctc_log_2.csv"  # ctc file with column wise data

log_df = pd.read_csv(log_file_path)

t = np.array(log_df['t'])
tau_delan_1 = np.array(log_df['tau_delan_1'])
tau_delan_2 = np.array(log_df['tau_delan_2'])
tau_delan_3 = np.array(log_df['tau_delan_3'])
tau_dnn_1 = np.array(log_df['tau_dnn_1'])
tau_dnn_2 = np.array(log_df['tau_dnn_2'])
tau_dnn_3 = np.array(log_df['tau_dnn_3'])
tau_fb_1 = np.array(log_df['tau_fb_1'])
tau_fb_2 = np.array(log_df['tau_fb_2'])
tau_fb_3 = np.array(log_df['tau_fb_3'])
tau_total_1 = np.array(log_df['tau_total_1'])
tau_total_2 = np.array(log_df['tau_total_2'])
tau_total_3 = np.array(log_df['tau_total_3'])
tau_sensed_1 = np.array(log_df['tau_sensed_1'])
tau_sensed_2 = np.array(log_df['tau_sensed_2'])
tau_sensed_3 = np.array(log_df['tau_sensed_3'])


ctc_log_df = pd.read_csv(ctc_log_file_path)

ctc_tau_model_1 = np.array(ctc_log_df['tau_model_1'])
ctc_tau_model_2 = np.array(ctc_log_df['tau_model_2'])
ctc_tau_model_3 = np.array(ctc_log_df['tau_model_3'])
ctc_tau_fb_1 = np.array(ctc_log_df['tau_fb_1'])
ctc_tau_fb_2 = np.array(ctc_log_df['tau_fb_2'])
ctc_tau_fb_3 = np.array(ctc_log_df['tau_fb_3'])
ctc_tau_total_1 = np.array(ctc_log_df['tau_total_1'])
ctc_tau_total_2 = np.array(ctc_log_df['tau_total_2'])
ctc_tau_total_3 = np.array(ctc_log_df['tau_total_3'])
ctc_tau_sensed_1 = np.array(ctc_log_df['tau_sensed_1'])
ctc_tau_sensed_2 = np.array(ctc_log_df['tau_sensed_2'])
ctc_tau_sensed_3 = np.array(ctc_log_df['tau_sensed_3'])


data = {}
with open(dataset_file_path, 'r') as f:
    reader = csv.reader(f)
    for row in reader:
        if row:
            key = row[0]
            values = [float(val) for val in row[1:]]
            data[key] = np.array(values)

# Extract relevant arrays
dataset_tau1 = data['tau1']
dataset_tau2 = data['tau2']
dataset_tau3 = data['tau3']



In [25]:
import matplotlib
matplotlib.use('TkAgg')   # or Qt5Agg

import matplotlib.pyplot as plt
import re


# Extract trajectory name from log_file_path
match = re.search(r'path_(\d+)_joint_states', log_file_path)
if match:
    trajectory_name = f"path_{match.group(1)}"
    output_filename = f'trajectory_comparison_{trajectory_name}_torques.png'
else:
    output_filename = 'trajectory_comparison_torques.png'

# Create figure with 3 subplots (one for each joint)
fig, axes = plt.subplots(3, 1, figsize=(14, 11))
fig.suptitle(f'Torque Comparison for All Joints - {trajectory_name if match else "Unknown"}', fontsize=16, fontweight='bold')

joints = [1, 2, 3]
torque_data = {
    'delan': [tau_delan_1, tau_delan_2, tau_delan_3],
    'dnn': [tau_dnn_1, tau_dnn_2, tau_dnn_3],
    'fb': [tau_fb_1, tau_fb_2, tau_fb_3],
    'total': [tau_total_1, tau_total_2, tau_total_3],
    'sensed': [tau_sensed_1, tau_sensed_2, tau_sensed_3],
    'ctc_model': [ctc_tau_model_1, ctc_tau_model_2, ctc_tau_model_3],
    # 'ctc_fb': [ctc_tau_fb_1, ctc_tau_fb_2, ctc_tau_fb_3],
    'ctc_total': [ctc_tau_total_1, ctc_tau_total_2, ctc_tau_total_3],
    # 'ctc_sensed': [ctc_tau_sensed_1, ctc_tau_sensed_2, ctc_tau_sensed_3],
    'dataset_tau': [dataset_tau1, dataset_tau2, dataset_tau3]
}

colors = {
    'delan': '#1f77b4',      # blue
    'dnn': '#2ca02c',        # green
    'fb': '#d62728',         # red
    'total': '#9467bd',      # purple
    'sensed': '#ff7f0e',      # orange
    'ctc_model': '#1f77b4',   # blue
    'ctc_fb': '#d62728',       # red
    'ctc_total': '#9467bd',    # purple
    'ctc_sensed': '#ff7f0e',   # orange
    'dataset_tau': '#8c564b'  # brown
}
# use different line styles for ctc data
line_styles = {
    'delan': '-',
    'dnn': '-',
    'fb': '-',
    'total': '-',
    'sensed': '-',
    'ctc_model': '--',
    'ctc_fb': '--',
    'ctc_total': '--',
    'ctc_sensed': '--',
    'dataset_tau': '-'
}

# Plot for each joint
for joint_idx, (ax, joint) in enumerate(zip(axes, joints)):
    for torque_type, torques in torque_data.items():
        ax.plot(t, torques[joint_idx], label=f'{torque_type.upper()}', 
                color=colors[torque_type], linestyle=line_styles[torque_type], linewidth=2, alpha=0.8)
    
    ax.set_xlabel('Time (s)', fontsize=11)
    ax.set_ylabel('Torque (Nm)', fontsize=11)
    ax.set_title(f'Joint {joint} Torques', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3, linestyle='--')


ax.legend(loc='best', fontsize=10, framealpha=0.9)

plt.tight_layout()
plt.show()